# Run the prototype's algorithms in your browser

This notebook accompanies the research proposal *Self-Evolving LLM Algorithm Discovery for Singapore's Maritime Operations*.
It runs the algorithms that our prototype generated on the **official benchmark instances**, checks every solution with
the **official checker**, and compares the result with the recorded per-instance results in `results/data/`.
It needs **no API key** and no local installation: run the cells from top to bottom (**Runtime > Run all** runs everything,
including the two longer optional steps).

| Step | What it does | Time on Colab |
|---|---|---|
| Setup | Clone the repository and install the dependencies | 1-2 min |
| FrontierOR 1 | Download the official data of one task | < 1 min |
| FrontierOR 2 | Run one algorithm on one official large instance (60 s limit) and check the solution | ~1 min |
| FrontierOR 3 (optional) | Run the task's whole portfolio on its five large instances | 5-25 min |
| CO-Bench 1 | Download the official data of one problem | < 1 min |
| CO-Bench 2 | Run one algorithm on one instance (10 s limit) and check the solution | < 1 min |
| CO-Bench 3 (optional) | Run the problem's whole portfolio on its first *N* instances | a few min |

The algorithms stop at a wall-clock time limit, so their results depend on the machine. Colab CPUs are usually slower
than the machines used for the recorded results, and a few objective values may differ slightly; each step prints this
run and the recorded value side by side. Many algorithms are randomised, so a single run of one algorithm can also differ
from run to run; the recorded results are the best result of the whole portfolio on each instance (step 3).

## Setup

In [ ]:
import base64, getpass, os, subprocess, sys
from pathlib import Path

REPO = "PJ-NTU/llm-algorithm-construction-results"
here = Path.cwd()
if (here / "prototype_colab.py").exists():                     # opened from notebooks/ in a local clone
    ROOT = here.parent
elif (here / "notebooks" / "prototype_colab.py").exists():     # opened from the repository root
    ROOT = here
else:                                                     # Colab: clone the repository
    ROOT = here / REPO.split("/")[1]
    if not ROOT.exists():
        url = "https://github.com/%s.git" % REPO
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        if subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(ROOT)], env=env,
                          capture_output=True).returncode != 0:
            # Only while the repository is private: clone with a GitHub token that can read it.
            # The token is passed in a request header, so it is neither stored nor printed.
            token = getpass.getpass("GitHub token with read access to %s (input hidden): " % REPO).strip()
            auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
            r = subprocess.run(["git", "-c", "http.extraHeader=Authorization: Basic " + auth,
                                "clone", "-q", "--depth", "1", url, str(ROOT)], env=env, capture_output=True, text=True)
            del token, auth
            if r.returncode != 0:
                raise SystemExit("Could not clone %s with this token. The token needs access to this repository "
                                 "with 'Contents: Read-only'." % REPO)
pkgs = ["pulp==3.3.2", "mip==1.17.6", "networkx==3.5", "huggingface_hub>=0.24"]
if sys.platform == "darwin" and os.uname().machine == "arm64":
    pkgs.append("cbcbox==2.929")                          # native CBC for Apple Silicon (not needed on Colab)
for cmd in ([sys.executable, "-m", "pip", "install", "-q", *pkgs], [sys.executable, str(ROOT / "setup_runtime.py")]):
    r = subprocess.run(cmd, capture_output=True, text=True)   # printed here: Colab hides child-process output
    if r.returncode != 0:
        raise SystemExit(r.stdout[-1500:] + r.stderr[-1500:])
print(r.stdout.strip())

sys.path.insert(0, str(ROOT / "notebooks"))
import prototype_colab as A
os.chdir(ROOT)
DATA = ROOT.parent / "benchmark-data"
print("Setup done. Repository:", ROOT)

## FrontierOR

The 20 tasks of the FrontierOR self-evolution subset; each is evaluated on five official large instances with 60 s per
algorithm and instance. Pick a task and run the cells in order.

### FrontierOR 1 - download the official data of one task

Fetches the task from the Hugging Face dataset `frontieror/FrontierOR` at revision `37ccd8b6`, the revision used for the recorded results.

In [ ]:
TASK = "13_nagy2015"  #@param ["01_adulyasak2015", "02_archetti2007", "03_armbruster2012", "04_bard2002", "05_bertsimas2022", "06_bront2009", "07_carvalho1999", "08_desaulniers2014", "09_desaulniers2016", "10_furini2021", "11_kobeaga2024", "12_mehrotra1996", "13_nagy2015", "14_pedersen2024", "15_pinnoi1997", "16_roberti2018", "17_rostami2021", "18_schwerdfeger2016", "19_wangk2020", "20_watermeyer2020"]
FO_DATA = DATA / "FrontierOR"
A.fo_download(TASK, FO_DATA)

### FrontierOR 2 - run one algorithm on one instance

By default this runs the algorithm that produced the recorded result on the chosen instance. The official instance is
converted by the task's adapter, solved within 60 s on one core, converted back and judged by the **official FrontierOR
checker**; the Gurobi reference is the value provided by FrontierOR.

For the 49 problems (18 FrontierOR tasks, 31 CO-Bench problems) whose portfolio was updated by continued evolution (`EVOLVED` in
`prototype_colab.py`), the recorded algorithm is no longer in the portfolio, so the whole current portfolio is
run on the instance instead.

In [ ]:
INSTANCE = "large_1"  #@param ["large_1", "large_2", "large_3", "large_4", "large_5"]
ALGORITHM = ""  #@param {type:"string"}
res = A.fo_run_one(TASK, INSTANCE, FO_DATA, algorithm=ALGORITHM or None)

### FrontierOR 3 (optional) - run the whole portfolio

Runs every algorithm of the task on the five large instances and keeps the best feasible result per instance, as in the
recorded results. Algorithms run in parallel, one per CPU core.

In [ ]:
portfolio = A.fo_run_portfolio(TASK, FO_DATA)

## CO-Bench

The 36 problems of CO-Bench; every instance is solved by every algorithm of the problem's portfolio with a 10 s limit.
The instance score is 1 if the objective equals or improves on the best-known value, and otherwise the ratio of the
smaller to the larger of the two values. For problems 4, 15, 19, 23 and 24 the official evaluator returns a ratio
(best-known value or lower bound divided by the objective); the notebook converts it back to the objective, as listed
in the recorded per-instance table.

### CO-Bench 1 - download the official data of one problem

Fetches only the chosen problem from the Hugging Face dataset `CO-Bench/CO-Bench` at the revision tested with the harness.

In [ ]:
PROBLEM = "01_aircraft_landing"  #@param ["01_aircraft_landing", "02_assignment_problem", "03_assortment_problem", "04_bin_packing_one_dimensional", "05_capacitated_warehouse_location", "06_common_due_date_scheduling", "07_constrained_guillotine_cutting", "08_constrained_non_guillotine_cutting", "09_container_loading", "10_container_loading_with_weight_restrictions", "11_corporate_structuring", "12_crew_scheduling", "13_equitable_partitioning_problem", "14_euclidean_steiner_problem", "15_flow_shop_scheduling", "16_generalised_assignment_problem", "17_graph_colouring", "18_hybrid_reentrant_shop_scheduling", "19_job_shop_scheduling", "20_maximal_independent_set", "21_multi_demand_multidimensional_knapsack_problem", "22_multidimensional_knapsack_problem", "23_open_shop_scheduling", "24_p_median_capacitated", "25_p_median_uncapacitated", "26_packing_unequal_circles", "27_packing_unequal_circles_area", "28_packing_unequal_rectangles_and_squares", "29_packing_unequal_rectangles_and_squares_area", "30_resource_constrained_shortest_path", "31_set_covering", "32_set_partitioning", "33_travelling_salesman_problem", "34_uncapacitated_warehouse_location", "35_vehicle_routing_period_routing", "36_unconstrained_guillotine_cutting"]
CB_DATA = DATA / "CO-Bench"
A.cb_download(PROBLEM, CB_DATA)

### CO-Bench 2 - run one algorithm on one instance

`INDEX` is the row of the problem's per-instance table (0 = first instance). The solution is checked by the prototype's checker and
by the **official CO-Bench evaluator**.

For the 49 problems (18 FrontierOR tasks, 31 CO-Bench problems) whose portfolio was updated by continued evolution (`EVOLVED` in
`prototype_colab.py`), the recorded algorithm is no longer in the portfolio, so the whole current portfolio is
run on the instance instead.

In [ ]:
INDEX = 0  #@param {type:"integer"}
ALGORITHM = ""  #@param {type:"string"}
one = A.cb_run_one(PROBLEM, INDEX, CB_DATA, algorithm=ALGORITHM or None)

### CO-Bench 3 (optional) - run the whole portfolio

Runs every algorithm of the problem on its first `MAX_INSTANCES` instances (0 = all) and compares the best result per
instance with the recorded results.

In [ ]:
MAX_INSTANCES = 10  #@param {type:"integer"}
best = A.cb_run_portfolio(PROBLEM, CB_DATA, max_instances=MAX_INSTANCES or None)